# 감성 분석 (Sentiment Analysis)

BERT를 활용한 텍스트 감성 분석 모델을 구축합니다.

## 학습 목표
1. Hugging Face Transformers 사용법
2. BERT 파인튜닝
3. 텍스트 분류 파이프라인 구축

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. 사전 학습된 감성 분석 파이프라인

In [ ]:
# 사전 학습된 감성 분석 모델
classifier = pipeline('sentiment-analysis')

# 테스트
texts = [
    "I love this movie! It's absolutely fantastic.",
    "This is the worst product I've ever bought.",
    "It's okay, nothing special but not bad either."
]

for text in texts:
    result = classifier(text)[0]
    print(f"Text: {text}")
    print(f"Sentiment: {result['label']}, Score: {result['score']:.4f}")
    print()

## 2. IMDB 데이터셋 로드

In [ ]:
# IMDB 데이터셋 로드
dataset = load_dataset('imdb')

print(f"Train: {len(dataset['train'])}")
print(f"Test: {len(dataset['test'])}")

# 샘플 확인
print("\n샘플 리뷰:")
print(f"Text: {dataset['train'][0]['text'][:200]}...")
print(f"Label: {dataset['train'][0]['label']} (0=negative, 1=positive)")

## 3. 토크나이저 및 전처리

In [ ]:
# 모델 및 토크나이저 로드
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 토크나이즈 함수
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

# 데이터셋 토크나이즈
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.rename_column('label', 'labels')
tokenized_datasets.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

In [ ]:
# 작은 서브셋 사용 (데모용)
small_train = tokenized_datasets['train'].shuffle(seed=42).select(range(1000))
small_test = tokenized_datasets['test'].shuffle(seed=42).select(range(200))

print(f"Train subset: {len(small_train)}")
print(f"Test subset: {len(small_test)}")

## 4. BERT 모델 파인튜닝

In [ ]:
# 분류 모델 로드
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# 평가 지표
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='binary')
    return {'accuracy': acc, 'f1': f1}

# 학습 설정
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics,
)

In [ ]:
# 학습
trainer.train()

In [ ]:
# 평가
eval_results = trainer.evaluate()
print(f"Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"F1 Score: {eval_results['eval_f1']:.4f}")

## 5. 모델 사용

In [ ]:
# 예측 함수
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=256)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
    
    sentiment = 'Positive' if pred == 1 else 'Negative'
    confidence = probs[0][pred].item()
    
    return sentiment, confidence

# 테스트
test_texts = [
    "This movie was amazing! Great acting and storyline.",
    "Terrible waste of time. I want my money back.",
    "The film was okay, but nothing memorable."
]

for text in test_texts:
    sentiment, confidence = predict_sentiment(text)
    print(f"Text: {text}")
    print(f"Prediction: {sentiment} ({confidence:.2%})")
    print()

## 연습 문제

1. 다른 BERT 변형 (RoBERTa, ALBERT)을 사용해보세요.
2. 한국어 감성 분석 (NSMC 데이터셋)을 시도해보세요.
3. Multi-class 감성 분류 (긍정/중립/부정)를 구현해보세요.